# TP 2 (Transformer) - Construire l'auto-attention pas a pas

**Cours NLP et Transformers - CY Tech**

Objectif : partir d'une simple moyenne des tokens precedents et arriver a une
tete d'attention qui apprend elle-meme quels tokens doivent contribuer.

Invariant a verifier a chaque etape : une ligne de poids contient la
contribution des tokens passes au calcul du token courant, et sa somme vaut
toujours 1.

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

## Etape 1 - Agreger par produit matriciel

Avant l'attention, on verifie qu'une ligne de matrice peut choisir et ponderer
plusieurs vecteurs. La matrice triangulaire normalisee realise une moyenne du
passe. Chaque ligne de `a` somme a 1, donc chaque ligne de `c` est une moyenne
des lignes de `b`.

In [2]:
a = torch.tril(torch.ones(3, 3))
a = a / a.sum(1, keepdim=True)
b = torch.randint(0, 10, (3, 2)).float()
c = a @ b

print(a)
print(b)
print(c)

tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
tensor([[5., 7.],
        [2., 0.],
        [5., 3.]])
tensor([[5.0000, 7.0000],
        [3.5000, 3.5000],
        [4.0000, 3.3333]])


## Etape 2 - Moyenner uniquement le passe

Pour chaque batch et chaque instant t, on construit la moyenne de tous les
tokens visibles jusqu'a t inclus. Aucune position ne doit acceder au futur.

In [3]:
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)

xbow = torch.zeros((B, T, C))
for indice_batch in range(B):
    for indice_temps in range(T):
        passe = x[indice_batch, :indice_temps + 1]
        xbow[indice_batch, indice_temps] = passe.mean(dim=0)

print("forme de x :", x.shape)
print("forme de xbow :", xbow.shape)

forme de x : torch.Size([4, 8, 2])
forme de xbow : torch.Size([4, 8, 2])


## Etape 3 - Vectoriser (version tril et version softmax)

La matrice triangulaire reproduit la moyenne causale. Le masque a moins l'infini
suivi de Softmax produit exactement les memes poids : Softmax d'une valeur a
moins l'infini vaut zero.

In [4]:
# Version 1 : moyenne par matrice triangulaire
poids = torch.tril(torch.ones(T, T))
poids = poids / poids.sum(1, keepdim=True)
xbow2 = poids @ x
print("tril egale la boucle :", torch.allclose(xbow, xbow2, atol=1e-6))

# Version 2 : masque a moins l'infini puis Softmax
tril = torch.tril(torch.ones(T, T))
poids = torch.zeros((T, T))
poids = poids.masked_fill(tril == 0, float("-inf"))
poids = F.softmax(poids, dim=-1)
xbow3 = poids @ x
print("softmax egale la boucle :", torch.allclose(xbow, xbow3, atol=1e-6))

tril egale la boucle : True
softmax egale la boucle : True


## Etape 4 - Calculer des poids d'attention appris

Les poids ne sont plus fixes a 1 / (t+1) : ils proviennent de la compatibilite
entre les queries et les keys. On masque le futur avant Softmax, puis on melange
les values. La sortie melange les values (information portee), pas les keys.

In [5]:
torch.manual_seed(1337)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)        # (B, T, head_size)
q = query(x)      # (B, T, head_size)
poids = q @ k.transpose(-2, -1)   # (B, T, T)

tril = torch.tril(torch.ones(T, T))
poids = poids.masked_fill(tril == 0, float("-inf"))
poids = F.softmax(poids, dim=-1)

v = value(x)
sortie = poids @ v    # (B, T, head_size)

print("forme de la sortie :", sortie.shape)
print("poids du premier batch :")
print(poids[0])
print("somme de la derniere ligne :", round(poids[0, -1].sum().item(), 4))

forme de la sortie : torch.Size([4, 8, 16])
poids du premier batch :
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)
somme de la derniere ligne : 1.0


## Etape 5 - Stabiliser Softmax

La variance du produit scalaire augmente avec `head_size`. Diviser par la racine
de `head_size` ramene les scores dans une zone ou Softmax reste informatif, au
lieu de devenir presque un choix unique.

In [6]:
k = torch.randn(B, T, head_size)
q = torch.randn(B, T, head_size)

brut = q @ k.transpose(-2, -1)
mis_a_l_echelle = brut * head_size ** -0.5

print("variance sans echelle :", round(brut.var().item(), 3))
print("variance avec echelle :", round(mis_a_l_echelle.var().item(), 3))

variance sans echelle : 17.469
variance avec echelle : 1.092


## Bilan

- Chaque ligne de `poids` somme a 1 : c'est une moyenne ponderee.
- Le masque empeche l'information de remonter du futur.
- Query, Key et Value jouent des roles distincts : la Query dit ce que le token
  cherche, la Key ce que chaque token peut apporter, la Value l'information
  reellement transmise.
- La mise a l'echelle par la racine de `head_size` stabilise Softmax.